<a href="https://colab.research.google.com/github/Luyzaluyza/olist-logistics-analysis/blob/main/olist-logistics-analysis/notebooks/cleaning%20/Limpeza_para_desigualdade_regional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Limpeza dos dados para relatorio de desigaldade regional

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/FIAP/primeiro/'

orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
customers = pd.read_csv(base_path + 'olist_customers_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
payments = pd.read_csv(base_path + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(base_path + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(base_path + 'product_category_name_translation.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
tabelas = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

for nome, df in tabelas.items():
    print(f'{nome}: {df.shape[0]} linhas e {df.shape[1]} colunas')

orders: 99441 linhas e 8 colunas
customers: 99441 linhas e 5 colunas
order_items: 112650 linhas e 7 colunas
payments: 103886 linhas e 5 colunas
reviews: 99224 linhas e 7 colunas
products: 32951 linhas e 9 colunas
sellers: 3095 linhas e 4 colunas
category_translation: 71 linhas e 2 colunas


In [ ]:
orders = orders.drop_duplicates().copy()
customers = customers.drop_duplicates().copy()
order_items = order_items.drop_duplicates().copy()
payments = payments.drop_duplicates().copy()
reviews = reviews.drop_duplicates().copy()
products = products.drop_duplicates().copy()
sellers = sellers.drop_duplicates().copy()
category_translation = category_translation.drop_duplicates().copy()

In [ ]:
datas_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datas_orders:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date'],
    errors='coerce'
)

reviews['review_creation_date'] = pd.to_datetime(
    reviews['review_creation_date'],
    errors='coerce'
)

reviews['review_answer_timestamp'] = pd.to_datetime(
    reviews['review_answer_timestamp'],
    errors='coerce'
)

In [ ]:
df_estados_regiao = pd.DataFrame({
    'customer_state': [
        'AC','AP','AM','PA','RO','RR','TO',
        'AL','BA','CE','MA','PB','PE','PI','RN','SE',
        'DF','GO','MT','MS',
        'ES','MG','RJ','SP',
        'PR','RS','SC'
    ],
    'estado': [
        'Acre','Amapá','Amazonas','Pará','Rondônia','Roraima','Tocantins',
        'Alagoas','Bahia','Ceará','Maranhão','Paraíba','Pernambuco','Piauí','Rio Grande do Norte','Sergipe',
        'Distrito Federal','Goiás','Mato Grosso','Mato Grosso do Sul',
        'Espírito Santo','Minas Gerais','Rio de Janeiro','São Paulo',
        'Paraná','Rio Grande do Sul','Santa Catarina'
    ],
    'regiao': [
        'Norte','Norte','Norte','Norte','Norte','Norte','Norte',
        'Nordeste','Nordeste','Nordeste','Nordeste','Nordeste','Nordeste','Nordeste','Nordeste','Nordeste',
        'Centro-Oeste','Centro-Oeste','Centro-Oeste','Centro-Oeste',
        'Sudeste','Sudeste','Sudeste','Sudeste',
        'Sul','Sul','Sul'
    ]
})

limpando as bases

In [ ]:
payments_clean = payments.groupby('order_id').agg(
    payment_value_total=('payment_value', 'sum'),
    payment_installments_max=('payment_installments', 'max'),
    payment_types=('payment_type', lambda x: ', '.join(sorted(x.dropna().unique())))
).reset_index()

display(payments_clean.head())

,order_id,payment_value_total,payment_installments_max,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card


In [ ]:
reviews_clean = reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean'),
    review_creation_date=('review_creation_date', 'min'),
    review_answer_timestamp=('review_answer_timestamp', 'max'),
    qtd_reviews=('review_id', 'nunique')
).reset_index()

display(reviews_clean.head())

,order_id,review_score,review_creation_date,review_answer_timestamp,qtd_reviews
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,2017-09-21,2017-09-22 10:57:03,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,2017-05-13,2017-05-15 11:34:13,1
2,000229ec398224ef6ca0657da4fc703e,5.0,2018-01-23,2018-01-23 16:06:31,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,2018-08-15,2018-08-15 16:39:01,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,2017-03-02,2017-03-03 10:54:59,1


base de pedidos limpa

In [ ]:
df_pedidos = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

df_pedidos = df_pedidos.merge(
    df_estados_regiao,
    on='customer_state',
    how='left'
)

df_pedidos = df_pedidos.merge(
    payments_clean,
    on='order_id',
    how='left'
)

df_pedidos = df_pedidos.merge(
    reviews_clean,
    on='order_id',
    how='left'
)

display(df_pedidos.head())
df_pedidos.info()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_state,estado,regiao,payment_value_total,payment_installments_max,payment_types,review_score,review_creation_date,review_answer_timestamp,qtd_reviews
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,SP,São Paulo,Sudeste,38.71,1.0,"credit_card, voucher",4.0,2017-10-11,2017-10-12 03:43:48,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,BA,Bahia,Nordeste,141.46,1.0,boleto,4.0,2018-08-08,2018-08-08 18:37:50,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,GO,Goiás,Centro-Oeste,179.12,3.0,credit_card,5.0,2018-08-18,2018-08-22 19:07:58,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,RN,Rio Grande do Norte,Nordeste,72.20,1.0,credit_card,5.0,2017-12-03,2017-12-05 19:21:58,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,SP,São Paulo,Sudeste,28.62,1.0,credit_card,5.0,2018-02-17,2018-02-18 13:02:51,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_unique_id             99441 non-null  object        
 9   customer_zip_code_prefix       99441 non-null  int64         
 10  customer_city                  99441 non-null  object        
 11  customer_state 

métricas logísticas

In [ ]:
df_pedidos['dias_entrega'] = (
    df_pedidos['order_delivered_customer_date'] -
    df_pedidos['order_purchase_timestamp']
).dt.days

df_pedidos['dias_estimados'] = (
    df_pedidos['order_estimated_delivery_date'] -
    df_pedidos['order_purchase_timestamp']
).dt.days

df_pedidos['dias_atraso'] = (
    df_pedidos['order_delivered_customer_date'] -
    df_pedidos['order_estimated_delivery_date']
).dt.days

df_pedidos['atrasado'] = df_pedidos['dias_atraso'] > 0

df_pedidos['ano_mes'] = df_pedidos['order_purchase_timestamp'].dt.to_period('M').astype(str)

display(df_pedidos.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_types,review_score,review_creation_date,review_answer_timestamp,qtd_reviews,dias_entrega,dias_estimados,dias_atraso,atrasado,ano_mes
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,"credit_card, voucher",4.0,2017-10-11,2017-10-12 03:43:48,1.0,8.0,15,-8.0,False,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,boleto,4.0,2018-08-08,2018-08-08 18:37:50,1.0,13.0,19,-6.0,False,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,credit_card,5.0,2018-08-18,2018-08-22 19:07:58,1.0,9.0,26,-18.0,False,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,credit_card,5.0,2017-12-03,2017-12-05 19:21:58,1.0,13.0,26,-13.0,False,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,credit_card,5.0,2018-02-17,2018-02-18 13:02:51,1.0,2.0,12,-10.0,False,2018-02


base de itens limpa

In [ ]:
df_itens = order_items.merge(
    products,
    on='product_id',
    how='left'
)

df_itens = df_itens.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

df_itens = df_itens.merge(
    sellers,
    on='seller_id',
    how='left'
)

display(df_itens.head())
df_itens.info()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 19 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  object        
 1   order_item_id                  112650 non-null  int64         
 2   product_id                     112650 non-null  object        
 3   seller_id                      112650 non-null  object        
 4   shipping_limit_date            112650 non-null  datetime64[ns]
 5   price                          112650 non-null  float64       
 6   freight_value                  112650 non-null  float64       
 7   product_category_name          111047 non-null  object        
 8   product_name_lenght            111047 non-null  float64       
 9   product_description_lenght     111047 non-null  float64       
 10  product_photos_qty             111047 non-null  float64       
 11  

In [ ]:
df_itens['product_volume_cm3'] = (
    df_itens['product_length_cm'] *
    df_itens['product_height_cm'] *
    df_itens['product_width_cm']
)

df_itens['product_category_name_final'] = df_itens['product_category_name_english'].fillna(
    df_itens['product_category_name']
)

display(df_itens.head())

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,product_volume_cm3,product_category_name_final
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,...,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP,3528.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,...,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP,60000.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,...,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG,14157.0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,...,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP,2400.0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,...,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR,42000.0,garden_tools


# Agregar as bases

In [ ]:
itens_por_pedido = df_itens.groupby('order_id').agg(
    qtd_itens=('order_item_id', 'count'),
    valor_produtos_total=('price', 'sum'),
    frete_total=('freight_value', 'sum'),
    peso_total_g=('product_weight_g', 'sum'),
    volume_total_cm3=('product_volume_cm3', 'sum'),
    categorias=('product_category_name_final', lambda x: ', '.join(sorted(x.dropna().unique()))),
    vendedores=('seller_id', lambda x: ', '.join(sorted(x.dropna().unique()))),
    estados_vendedores=('seller_state', lambda x: ', '.join(sorted(x.dropna().unique())))
).reset_index()

display(itens_por_pedido.head())

,order_id,qtd_itens,valor_produtos_total,frete_total,peso_total_g,volume_total_cm3,categorias,vendedores,estados_vendedores
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,650.0,3528.0,cool_stuff,48436dade18ac8b2bce089ec2a041202,SP
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,30000.0,60000.0,pet_shop,dd7ddc04e1b6c2c614352b383efe2d36,SP
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,3050.0,14157.0,furniture_decor,5b51032eddd242adc84c38acab88f23d,MG
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,200.0,2400.0,perfumery,9d7a1d34a5052409006425275ba1c2b4,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,3750.0,42000.0,garden_tools,df560393f3a51e74553ab94004ba5c87,PR


In [ ]:
df_pedidos_final = df_pedidos.merge(
    itens_por_pedido,
    on='order_id',
    how='left'
)

display(df_pedidos_final.head())
df_pedidos_final.info()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,atrasado,ano_mes,qtd_itens,valor_produtos_total,frete_total,peso_total_g,volume_total_cm3,categorias,vendedores,estados_vendedores
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,False,2017-10,1.0,29.99,8.72,500.0,1976.0,housewares,3504c0cb71d7fa48d967e0e4c94d59d9,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,False,2018-07,1.0,118.70,22.76,400.0,4693.0,perfumery,289cdb325fb7e7f891c38608bf9e0962,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,False,2018-08,1.0,159.90,19.22,420.0,9576.0,auto,4869f7a5dfa277a7dca6462dcf3b52b2,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,False,2017-11,1.0,45.00,27.20,450.0,6000.0,pet_shop,66922902710d126a0e7d26b0e3805106,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,False,2018-02,1.0,19.90,8.72,250.0,11475.0,stationery,2c9e548be18521d1c43cde1c582c6de8,SP


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 34 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_unique_id             99441 non-null  object        
 9   customer_zip_code_prefix       99441 non-null  int64         
 10  customer_city                  99441 non-null  object        
 11  customer_state 

# Filtro por periodo que vamos ultilizar 2017 - 2018

A análise foi restrita ao período de 2017 a 2018, representando 99.67% da base total, garantindo consistência temporal dos dados.

In [ ]:
df_pedidos_final['order_purchase_timestamp'] = pd.to_datetime(
    df_pedidos_final['order_purchase_timestamp'],
    errors='coerce'
)

total_pedidos = df_pedidos_final.shape[0]

# filtro 2017-2018
df_pedidos_2017_2018 = df_pedidos_final[
    (df_pedidos_final['order_purchase_timestamp'] >= '2017-01-01') &
    (df_pedidos_final['order_purchase_timestamp'] <= '2018-12-31')
].copy()

total_filtrado = df_pedidos_2017_2018.shape[0]

percentual = (total_filtrado / total_pedidos) * 100

print(f"Total original: {total_pedidos}")
print(f"Total 2017-2018: {total_filtrado}")
print(f"Percentual da base: {percentual:.2f}%")

Total original: 99441
Total 2017-2018: 99112
Percentual da base: 99.67%


In [ ]:
#qualidade final da base

In [ ]:
print("Quantidade de pedidos únicos:", df_pedidos_2017_2018['order_id'].nunique())
print("Quantidade de linhas:", df_pedidos_2017_2018.shape[0])

print("\nStatus dos pedidos:")
display(df_pedidos_2017_2018['order_status'].value_counts())

print("\nNulos principais:")
display(df_pedidos_2017_2018[
    [
        'order_id',
        'order_status',
        'customer_state',
        'regiao',
        'dias_entrega',
        'dias_atraso',
        'valor_produtos_total',
        'frete_total',
        'review_score'
    ]
].isnull().sum())

Quantidade de pedidos únicos: 99112
Quantidade de linhas: 99112

Status dos pedidos:


,count
order_status,
delivered,96211
shipped,1098
unavailable,602
canceled,599
processing,299
invoiced,296
created,5
approved,2



Nulos principais:


,0
order_id,0
order_status,0
customer_state,0
regiao,0
dias_entrega,2908
dias_atraso,2908
valor_produtos_total,758
frete_total,758
review_score,763


In [ ]:
df_pedidos_entregues = df_pedidos_2017_2018[
    df_pedidos_2017_2018['order_status'] == 'delivered'
].copy()

df_pedidos_problema = df_pedidos_2017_2018[
    df_pedidos_2017_2018['order_status'].isin(['canceled', 'unavailable'])
].copy()

print("Pedidos entregues:", df_pedidos_entregues.shape)
print("Pedidos cancelados/indisponíveis:", df_pedidos_problema.shape)

Pedidos entregues: (96211, 34)
Pedidos cancelados/indisponíveis: (1201, 34)


# Exportar arquivos limpos

In [ ]:
df_pedidos_2017_2018.to_csv(base_path + 'df_pedidos_limpo_2017_2018.csv', index=False)
df_pedidos_entregues.to_csv(base_path + 'df_pedidos_entregues_limpo.csv', index=False)
df_pedidos_problema.to_csv(base_path + 'df_pedidos_problema_limpo.csv', index=False)
df_itens.to_csv(base_path + 'df_itens_limpo.csv', index=False)

print("Arquivos exportados com sucesso!")

Arquivos exportados com sucesso!


In [ ]:
df_entregues = pd.read_csv(base_path + 'df_pedidos_entregues_limpo.csv')
df_problema = pd.read_csv(base_path + 'df_pedidos_problema_limpo.csv')
df_itens = pd.read_csv(base_path + 'df_itens_limpo.csv')